
# vLLM Source-Level Project

**Today's goal:** Environment → Run → Serve → Measure → Locate source.

You are **not** optimizing vLLM today. By the end of the notebook you should be able to:

1. Confirm the Colab T4 environment.
2. Install and record the exact vLLM/PyTorch/CUDA environment.
3. Run offline inference through `vllm.LLM`.
4. Inspect `RequestOutput`.
5. Run a very small latency experiment.
6. Start the OpenAI-compatible vLLM server.
7. Send one request through the HTTP serving path.
8. Record observations for Day 2.

### Rules
- Cells marked **TODO — YOU WRITE THIS** are intentionally incomplete.
- Try not to look up the final implementation until you have attempted it.
- Do not optimize anything yet.
- Use a **T4 runtime**: Runtime → Change runtime type → T4 GPU.


## 0. Confirm the GPU

In [1]:

!nvidia-smi


Sat Sep 19 12:16:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----


### Checkpoint 0
Before continuing, answer in your notes:

- GPU model:
- Total VRAM:
- Driver version:
- CUDA version reported by `nvidia-smi`:


## 1. Install vLLM and helper packages

In [2]:
!pip uninstall -y torchaudio
!pip install -U torchaudio==2.11.0+cu130 \
  --index-url https://download.pytorch.org/whl/cu130

Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
Looking in indexes: https://download.pytorch.org/whl/cu130
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 34.8 MB/s eta 0:00:00


In [3]:
# vLLM changes quickly. For Day 1 we use the released package and record
# the exact installed version below so future benchmarks are reproducible.
%pip install -q -U vllm openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.0/316.0 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 117.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 96.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.7/322.7 kB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 94.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 109.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43

In [ ]:
import os
os.kill(os.getpid(), 9)


> If Colab asks you to restart the runtime after installation, restart it and then rerun from Cell 0.


## 2. Record the experiment environment

In [ ]:
import sys
import torch
import vllm
import time
import statistics

print("Python :", sys.version)
print("PyTorch:", torch.__version__)
print("Torch CUDA build:", torch.version.cuda)
print("vLLM   :", vllm.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU:", props.name)
    print(f"VRAM: {props.total_memory / 1024**3:.2f} GiB")


## 3. Offline inference — first core exercise

In [1]:
import torch
import torchaudio

print("torch:", torch.__version__)
print("torch CUDA:", torch.version.cuda)
print("torchaudio:", torchaudio.__version__)

torch: 2.13.0+cu130
torch CUDA: 13.0
torchaudio: 2.11.0+cu130


In [2]:
%%writefile /content/test_vllm.py
from vllm import LLM, SamplingParams
import time
import statistics

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_MODEL_LEN = 4096
GPU_MEMORY_UTILIZATION = 0.70

output_path = "/content/request_output_inspection.txt"

prompts = [
    "Explain in one sentence what a KV cache is in LLM inference.",
    "Why can continuous batching improve GPU utilization?",
]

# ============================================================
# TODO — YOU WRITE THIS
#
# Task A: Construct SamplingParams.
# Requirements:
#   - temperature = 0.0
#   - max_tokens = 64
#
# sampling_params = ...
# ============================================================

sampling_params = SamplingParams(temperature=0.0,
                                 max_tokens=64)

# ============================================================
# TODO — YOU WRITE THIS
#
# Task B: Construct the vLLM LLM object.
#
# Requirements:
#   model=MODEL
#   dtype="float16"
#   max_model_len=MAX_MODEL_LEN
#   gpu_memory_utilization=GPU_MEMORY_UTILIZATION
#
# llm = ...
# ============================================================

llm = LLM(model = MODEL,
          dtype = "float16",
          max_model_len = MAX_MODEL_LEN,
          gpu_memory_utilization = GPU_MEMORY_UTILIZATION)

# ============================================================
# TODO — YOU WRITE THIS
#
# Task C: Generate outputs for `prompts`.
#
# Ask yourself:
#   1. Which object owns the generate() method?
#   2. What two main arguments does it need here?
#
# outputs = ...
# ============================================================

outputs = llm.generate(
    prompts,
    sampling_params
)


# ============================================================
# Inspect RequestOutput
# Small offline timing experiment:
# ============================================================

short_prompt = "Briefly explain GPU memory bandwidth."
long_prompt = ("Explain the relationship between GPU compute throughput, memory "
               "bandwidth, arithmetic intensity, and kernel performance. " * 80)

def run_once(prompt):
    # ========================================================
    # TODO — YOU WRITE THIS
    #
    # Measure wall-clock latency around ONE llm.generate call.
    # Return:
    #   latency_seconds, output_token_count
    #
    # Notes:
    # - This is NOT yet a rigorous serving benchmark.
    # - llm.generate is offline/batched inference, not HTTP TTFT.
    # - We only want a Day-1 baseline and familiarity with outputs.
    # ========================================================
    # raise NotImplementedError
    start_time = time.perf_counter()
    output = llm.generate(
        [prompt],
        sampling_params
    )
    end_time = time.perf_counter()
    latency_seconds = end_time - start_time
    output_token_count = len(output[0].outputs[0].token_ids)
    return latency_seconds, output_token_count

# Warm-up: complete this after run_once() works.
# _ = run_once(short_prompt)

# ============================================================
# TODO — YOU WRITE THIS
#
# Run each prompt 3 times and report:
#   mean latency
#   mean output-token count
#
# Compare short_prompt vs long_prompt.
# ============================================================
short_results = [run_once(short_prompt) for _ in range(3)]
long_results = [run_once(long_prompt) for _ in range(3)]

short_latencies = [x[0] for x in short_results]
short_tokens = [x[1] for x in short_results]

long_latencies = [x[0] for x in long_results]
long_tokens = [x[1] for x in long_results]

with open(output_path, "w", encoding="utf-8") as f:

    # ========================================================
    # RequestOutput inspection
    # ========================================================
    f.write("=== RequestOutput Inspection ===\n\n")

    for output in outputs:
        candidate = output.outputs[0]

        f.write(f"Request ID: {output.request_id}\n")
        f.write(f"Prompt: {output.prompt}\n")
        f.write(f"Prompt token count: {len(output.prompt_token_ids)}\n")
        f.write(f"Finished: {output.finished}\n")

        f.write(f"Generated text: {candidate.text}\n")
        f.write(f"Generated token IDs: {candidate.token_ids}\n")
        f.write(f"Generated token count: {len(candidate.token_ids)}\n")
        f.write(f"Finish reason: {candidate.finish_reason}\n")

        f.write("-" * 60 + "\n")

    # ========================================================
    # Latency benchmark
    # ========================================================
    f.write("\n=== Offline Latency Benchmark ===\n\n")

    f.write("Short prompt:\n")
    f.write(f"Mean latency: {statistics.mean(short_latencies):.4f} s\n")
    f.write(f"Mean output tokens: {statistics.mean(short_tokens):.2f}\n")
    f.write(f"Raw latencies: {short_latencies}\n")
    f.write("\n")

    f.write("Long prompt:\n")
    f.write(f"Mean latency: {statistics.mean(long_latencies):.4f} s\n")
    f.write(f"Mean output tokens: {statistics.mean(long_tokens):.2f}\n")
    f.write(f"Raw latencies: {long_latencies}\n")

Writing /content/test_vllm.py


In [3]:
!python /content/test_vllm.py

INFO 09-19 12:22:11 [api_utils.py:286] non-default args: {'dtype': 'float16', 'max_model_len': 4096, 'gpu_memory_utilization': 0.7, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-1.5B-Instruct'}
config.json: 100% 660/660 [00:00<00:00, 3.43MB/s]
INFO 09-19 12:22:30 [model.py:684] Resolved architecture: Qwen2ForCausalLM
WARNING 09-19 12:22:30 [model.py:2355] Casting torch.bfloat16 to torch.float16.
INFO 09-19 12:22:30 [model.py:2021] Using max model len 4096
INFO 09-19 12:22:30 [scheduler.py:277] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 09-19 12:22:30 [kernel.py:369] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 26.4MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 77.4MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 124MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 151MB/s]
generation_config.json: 100% 242/242 [00:00<00:00,


### What you should notice

The important conceptual call today is:

```text
Python program
    ↓
LLM.generate(...)
    ↓
vLLM engine
    ↓
scheduler / KV cache / model runner
    ↓
GPU
    ↓
RequestOutput
```

Today we treat the middle as a black box. Day 2 starts opening it.


## 4. Inspect `RequestOutput`

## 5. Small offline timing experiment:


### Checkpoint 1 — write down what happened

Do **not** over-interpret the numbers yet.

| Workload | Mean latency | Mean generated tokens |
|---|---:|---:|
| short prompt | | |
| long prompt | | |

Questions:

1. Did the longer prompt increase latency?
2. Why might that happen?
3. Is this number TTFT? Why or why not?
4. Which stage should be more affected by prompt length: prefill or decode?


## 6. Inspect GPU memory after model load

In [6]:

print(torch.cuda.memory_summary(abbreviated=True))
!nvidia-smi


|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |      0 B   |      0 B   |      0 B   |      0 B   |
|---------------------------------------------------------------------------|
| Active memory         |      0 B   |      0 B   |      0 B   |      0 B   |
|---------------------------------------------------------------------------|
| Requested memory      |      0 B   |      0 B   |      0 B   |      0 B   |
|---------------------------------------------------------------


### Checkpoint 2

Record:

- GPU memory used after model load:
- Approximate free memory:
- Why does vLLM reserve significant GPU memory beyond model weights?
- What future structure do you expect to occupy much of that memory?

Do not worry if you cannot fully explain the last two yet — that is Day 4.



## 7. Release the offline engine before starting the server

A single T4 cannot comfortably host two copies of the same vLLM model.
Delete the offline engine and clear Python's references before starting a separate server process.


In [7]:

import gc

# Keep this infrastructure code as-is.
# del outputs
# del llm
gc.collect()
torch.cuda.empty_cache()

!nvidia-smi


Sat Sep 19 12:25:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             14W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----


## 8. Start the OpenAI-compatible vLLM server

The server uses the same model but exposes it over HTTP.

This cell is infrastructure, so it is provided for you.

If it fails because a CLI flag changed in your installed vLLM version, inspect:

```bash
vllm serve --help
```


In [8]:

import subprocess
import time
import os
import signal

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_MODEL_LEN = 4096
GPU_MEMORY_UTILIZATION = 0.70

server_log = open("/content/vllm_server.log", "w")

server = subprocess.Popen(
    [
        "vllm", "serve", MODEL,
        "--dtype", "float16",
        "--max-model-len", str(MAX_MODEL_LEN),
        "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),
        "--host", "127.0.0.1",
        "--port", "8000",
    ],
    stdout=server_log,
    stderr=subprocess.STDOUT,
)

print("Server PID:", server.pid)
print("Log: /content/vllm_server.log")


Server PID: 4186
Log: /content/vllm_server.log


## 9. Wait until the server is healthy

In [9]:

import requests
import time

for i in range(60):
    try:
        r = requests.get("http://127.0.0.1:8000/health", timeout=2)
        if r.status_code == 200:
            print("Server is ready.")
            break
    except Exception:
        pass

    if i % 5 == 0:
        print(f"Waiting... ({i})")
    time.sleep(2)
else:
    print("Server did not become healthy. Inspect the log:")
    print(open("/content/vllm_server.log").read()[-8000:])


Waiting... (0)
Waiting... (5)
Waiting... (10)
Waiting... (15)
Waiting... (20)
Waiting... (25)
Waiting... (30)
Waiting... (35)
Waiting... (40)
Waiting... (45)
Waiting... (50)
Waiting... (55)
Server did not become healthy. Inspect the log:
.67it/s]
Capturing CUDA graphs (PIECEWISE): 100%|██████████| 51/51 [00:03<00:00, 13.57it/s]
(EngineCore pid=4315) 
Capturing CUDA graphs (FULL): 100%|██████████| 35/35 [00:01<00:00, 20.26it/s]
(EngineCore pid=4315) INFO 09-19 12:27:44 [model_runner.py:960] Graph capturing finished in 6 secs, took 0.15 GiB
(EngineCore pid=4315) INFO 09-19 12:27:44 [gpu_worker.py:797] CUDA graph pool memory: 0.15 GiB (actual), 0.34 GiB (estimated), difference: 0.19 GiB (127.3%).
(EngineCore pid=4315) INFO 09-19 12:27:44 [gpu_worker.py:860] Free memory on device (14.46/14.56 GiB) on startup. Desired GPU memory utilization is (0.7, 10.19 GiB). Actual usage is 3.66 GiB for consumed memory (weights + non-torch), 0.78 GiB for peak activation, and 0.15 GiB for CUDAGraph memory

## 10. Send your first HTTP request — second core exercise

In [10]:

from openai import OpenAI

client = OpenAI(
    base_url="http://127.0.0.1:8000/v1",
    api_key="EMPTY",
)

# ============================================================
# TODO — YOU WRITE THIS
#
# Send ONE chat completion request.
#
# Requirements:
#   model=MODEL
#   one user message asking:
#       "What problem does PagedAttention solve?"
#   temperature=0.0
#   max_tokens=64
#
# response = client.chat.completions.create(...)
# ============================================================

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": "What problem does PagedAttention solve?"
        }
    ],
    temperature=0.0,
    max_tokens=64
)

# ============================================================
# TODO — YOU WRITE THIS
#
# Print only the generated assistant text.
# Explore `response` if you don't know the object layout.
# ============================================================
print(response.choices[0].message.content)

I'm sorry, but I couldn't find any information about a specific problem called "PagedAttention" that needs to be solved. It's possible that there might be some confusion or typo in the name.

Could you please provide more details or context about what kind of problem you're referring to? This will help me



## 11. Observe streaming and approximate TTFT

This is the first place today where you should distinguish:

- **TTFT**: request sent → first generated token/chunk arrives
- **E2E latency**: request sent → generation finishes

For Day 1, measuring Python streaming-chunk arrival time is sufficient.
It is not yet our final benchmark methodology.


In [11]:

import time

# ============================================================
# TODO — YOU WRITE THIS
#
# Send a streaming Chat Completions request.
#
# Measure:
#   start_time
#   first_nonempty_content_time
#   finish_time
#
# Compute:
#   approximate_TTFT = first_nonempty_content_time - start_time
#   E2E_latency      = finish_time - start_time
#
# Requirements:
#   prompt: "Explain continuous batching in about 100 words."
#   temperature=0.0
#   max_tokens=128
#   stream=True
#
# Important:
# Some initial stream chunks may contain metadata/role with no text.
# Count TTFT at the FIRST NON-EMPTY generated content chunk.
# ============================================================
start_time = time.perf_counter()
first_nonempty_content_time = None

stream = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": "Explain continuous batching in about 100 words."
        }
    ],
    temperature=0.0,
    max_tokens=128,
    stream=True
)

generated_text = ""

for chunk in stream:
    content = chunk.choices[0].delta.content

    if content:
        if first_nonempty_content_time is None:
            first_nonempty_content_time = time.perf_counter()

        generated_text += content
        print(content, end="", flush=True)

finish_time = time.perf_counter()

approximate_TTFT = first_nonempty_content_time - start_time
E2E_latency = finish_time - start_time

print("\n")
print(f"Approximate TTFT: {approximate_TTFT:.4f} s")
print(f"E2E latency:      {E2E_latency:.4f} s")

Continuous Batching is an optimization technique used in machine learning and data processing pipelines to improve efficiency and reduce latency. It involves dividing the input data into smaller batches that can be processed concurrently without waiting for previous batches to complete.

In traditional batch processing, all data is loaded into memory at once before being processed. This approach works well when dealing with small datasets or when there's sufficient computational resources available. However, it becomes inefficient as the size of the dataset grows due to increased memory usage and slower performance bottlenecks.

Continuous Batching addresses these issues by breaking down large datasets into multiple smaller batches during each iteration of training or processing. Each batch

Approximate TTFT: 0.0809 s
E2E latency:      2.0031 s



### Checkpoint 3

Record:

- Approximate TTFT:
- End-to-end latency:
- Generated output tokens (if you counted them):
- Why is TTFT different from total latency?
- Which part of inference dominates TTFT for a long prompt?


## 12. Locate the installed vLLM source tree

In [12]:

import inspect
import os
import vllm

vllm_root = os.path.dirname(inspect.getfile(vllm))
print("Installed vLLM source:", vllm_root)

# We only LOCATE files today. Do not dive deeply into them yet.
!find "$vllm_root/v1" -maxdepth 3 -type f | grep -E "(engine|scheduler|kv_cache|model_runner)" | head -80


Installed vLLM source: /usr/local/lib/python3.13/dist-packages/vllm
/usr/local/lib/python3.13/dist-packages/vllm/v1/engine/exceptions.py
/usr/local/lib/python3.13/dist-packages/vllm/v1/engine/__pycache__/detokenizer.cpython-313.pyc
/usr/local/lib/python3.13/dist-packages/vllm/v1/engine/__pycache__/utils.cpython-313.pyc
/usr/local/lib/python3.13/dist-packages/vllm/v1/engine/__pycache__/exceptions.cpython-313.pyc
/usr/local/lib/python3.13/dist-packages/vllm/v1/engine/__pycache__/__init__.cpython-313.pyc
/usr/local/lib/python3.13/dist-packages/vllm/v1/engine/__pycache__/core.cpython-313.pyc
/usr/local/lib/python3.13/dist-packages/vllm/v1/engine/__pycache__/parallel_sampling.cpython-313.pyc
/usr/local/lib/python3.13/dist-packages/vllm/v1/engine/__pycache__/output_processor.cpython-313.pyc
/usr/local/lib/python3.13/dist-packages/vllm/v1/engine/__pycache__/core_client.cpython-313.pyc
/usr/local/lib/python3.13/dist-packages/vllm/v1/engine/__pycache__/async_llm.cpython-313.pyc
/usr/local/lib/p


## 13. Day-1 source map — fill this yourself

Without trying to understand every line, locate likely files/classes for:

```text
User / API
    ↓
LLM / serving frontend
    ↓
Engine / EngineCore
    ↓
Scheduler
    ↓
KV-cache management
    ↓
Model runner
    ↓
GPU
```

Fill in:

| Layer | File / class you found | What you THINK it does |
|---|---|---|
| Offline frontend | | |
| Engine / core | | |
| Scheduler | | |
| KV cache | | |
| Model runner | | |

It is okay if some guesses are wrong. Day 2 will verify the request path.


## 14. Stop the server

In [13]:

# Infrastructure cleanup.
if server.poll() is None:
    server.terminate()
    try:
        server.wait(timeout=10)
    except subprocess.TimeoutExpired:
        server.kill()

server_log.close()
print("Server stopped.")


Server stopped.


# Day 2 — Tracing the vLLM Request Lifecycle

## Objective

The goal of Day 2 is to open the `LLM.generate()` black box and understand how a user prompt becomes an internal vLLM request before being scheduled for GPU execution.

Day 1 established that inference works correctly through vLLM. Day 2 focuses on the control flow behind that API call.

The main path to trace is:

```text
User Prompt
    ↓
LLM.generate()
    ↓
Internal completion path
    ↓
Request creation / preprocessing
    ↓
Engine / EngineCore
    ↓
Scheduler.add_request()
    ↓
Waiting queue
    ↓
Scheduler.schedule()
```

The goal is not to understand every line of vLLM source code. Instead, the focus is to identify the main objects, functions, and state transitions that a request passes through.

---

## What We Want to Understand

By the end of Day 2, we should be able to answer the following questions:

1. Where is `LLM.generate()` implemented?
2. Which internal function does `generate()` call next?
3. How is a user prompt converted into an internal request?
4. What information is stored inside the request object?
5. How does the request reach `EngineCore`?
6. How is the request inserted into the scheduler?
7. When does the request enter the waiting queue?
8. What does `Scheduler.schedule()` produce for the next execution step?

---

## Mental Model

At the beginning of Day 2, the system still looks like this:

```text
llm.generate(...)
      ↓
   black box
      ↓
RequestOutput
```

The purpose of Day 2 is to expand the black box into:

```text
llm.generate(...)
      ↓
LLM.generate()
      ↓
internal completion path
      ↓
request preprocessing
      ↓
internal request object
      ↓
EngineCore
      ↓
Scheduler.add_request()
      ↓
waiting queue
      ↓
Scheduler.schedule()
      ↓
scheduled workload
```

This request lifecycle will become the foundation for understanding continuous batching and scheduler behavior in later stages of the project.

---

## Source-Level Goals

During Day 2, we will use Python introspection tools such as:

```python
inspect.getfile(...)
inspect.getsource(...)
```

to locate and inspect the vLLM implementation currently installed in the environment.

The purpose is to follow the actual installed vLLM version rather than relying only on documentation or diagrams.

We will progressively trace:

```text
LLM.generate()
→ internal helper functions
→ request creation
→ engine submission
→ scheduler admission
```

At each step, we will record:

* source file
* class or function name
* input object
* output object
* role in the request lifecycle

---

## Request Lifecycle Table

As we trace the source code, we will gradually fill in the following table:

| Stage                 | File / Class | Function         | Purpose                                                              |
| --------------------- | ------------ | ---------------- | -------------------------------------------------------------------- |
| Public API            |              | `LLM.generate()` | Entry point for offline inference                                    |
| Completion path       |              |                  | Converts public API call into internal processing                    |
| Request preprocessing |              |                  | Converts prompt and sampling parameters into internal representation |
| Engine                |              |                  | Submits request to execution core                                    |
| EngineCore            |              |                  | Coordinates scheduling and model execution                           |
| Scheduler admission   |              | `add_request()`  | Inserts request into scheduler                                       |
| Waiting queue         |              |                  | Holds requests not currently executing                               |
| Scheduling            |              | `schedule()`     | Selects work for the next engine step                                |

---

## What We Are NOT Doing Today

Day 2 is about request flow, not scheduler optimization.

We will not yet deeply analyze:

* token budgets
* `num_computed_tokens`
* continuous batching policy
* chunked prefill policy
* KV-cache block allocation
* preemption
* scheduler optimization

Those topics will be handled in later stages.

The important goal today is simply:

> Understand how a prompt becomes a scheduler-visible request.

---

## Expected Output

At the end of Day 2, we should have a request lifecycle diagram similar to:

```text
User Prompt
    ↓
LLM.generate()
    ↓
Request preprocessing
    ↓
Internal Request
    ↓
EngineCore
    ↓
Scheduler.add_request()
    ↓
WAITING
    ↓
Scheduler.schedule()
    ↓
RUNNING / Scheduled Work
```

We should also be able to explain this process without looking at the source code.

---

## Day 2 Summary Goal

**Trace a request from `LLM.generate()` into the vLLM engine and scheduler, identifying the main source files, internal request representation, and queue transitions involved in request admission.**


In [14]:
"""
Day 2 — Tracing the vLLM Request Lifecycle

Goal:
    Trace the control flow from LLM.generate() toward the internal
    request-processing path and scheduler entry points.

This script focuses on source inspection rather than performance.
"""

import inspect

from vllm import LLM


def print_section(title):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)


# =============================================================================
# Step 1 — Locate LLM.generate()
# =============================================================================

print_section("Step 1 — Locate LLM.generate()")

# TODO:
# Use inspect.getfile(...) to find the source file that contains LLM.generate().
#
# Expected concept:
#     public API method
#         ↓
#     actual Python source file
#
# Hint:
#     inspect.getfile(...)
#
generate_file = inspect.getfile(LLM.generate)

print("LLM.generate() source file:")
print(generate_file)


Step 1 — Locate LLM.generate()
LLM.generate() source file:
/usr/local/lib/python3.13/dist-packages/vllm/entrypoints/llm.py


In [15]:
# =============================================================================
# Step 2 — Inspect the source code of LLM.generate()
# =============================================================================

print_section("Step 2 — Inspect LLM.generate() source")

# TODO:
# Use inspect.getsource(...) to retrieve the implementation of LLM.generate().
#
# After printing the source, identify:
#   1. What function does generate() eventually call?
#   2. Which arguments are forwarded?
#
generate_source = inspect.getsource(LLM.generate)

print(generate_source)


Step 2 — Inspect LLM.generate() source
    def generate(
        self,
        prompts: PromptType | Sequence[PromptType],
        sampling_params: SamplingParams | Sequence[SamplingParams] | None = None,
        *,
        use_tqdm: bool | Callable[..., tqdm] = True,
        lora_request: Sequence[LoRARequest] | LoRARequest | None = None,
        priority: list[int] | None = None,
        tokenization_kwargs: dict[str, Any] | None = None,
        mm_processor_kwargs: dict[str, Any] | None = None,
    ) -> list[RequestOutput]:
        """Generates the completions for the input prompts.

        This class automatically batches the given prompts, considering
        the memory constraint. For the best performance, put all of your prompts
        into a single list and pass it to this method.

        Args:
            prompts: The prompts to the LLM. You may pass a sequence of prompts
                for batch inference. See [PromptType][vllm.inputs.PromptType]
                for more

In [16]:
# =============================================================================
# Step 3 — Find the next internal method
# =============================================================================

print_section("Step 3 — Inspect the next internal completion method")

# TODO:
# Based on the source code from Step 2, identify the next important
# internal method called by generate().
#
# Example reasoning:
#
#     LLM.generate()
#         ↓
#     some_internal_method(...)
#
# Replace the placeholder below with the correct bound method.
#
next_method = LLM._run_completion

# TODO:
# Print the source file of the next method.
#
next_method_file = inspect.getfile(next_method)

print("Next method source file:")
print(next_method_file)


# TODO:
# Print the source code of the next method.
#
next_method_source = inspect.getsource(next_method)

print(next_method_source)


Step 3 — Inspect the next internal completion method
Next method source file:
/usr/local/lib/python3.13/dist-packages/vllm/entrypoints/offline_utils.py
    def _run_completion(
        self,
        prompts: PromptType | Sequence[PromptType],
        params: SamplingParams
        | PoolingParams
        | Sequence[SamplingParams | PoolingParams],
        output_type: type[_O],
        *,
        use_tqdm: bool | Callable[..., tqdm] = True,
        lora_request: Sequence[LoRARequest] | LoRARequest | None = None,
        priority: list[int] | None = None,
        tokenization_kwargs: dict[str, Any] | None = None,
        mm_processor_kwargs: dict[str, Any] | None = None,
    ):
        self._add_completion_requests(
            prompts=prompts,
            params=params,
            use_tqdm=use_tqdm,
            lora_request=lora_request,
            priority=priority,
            tokenization_kwargs=tokenization_kwargs,
            mm_processor_kwargs=mm_processor_kwargs,
        )
 

In [18]:
# =============================================================================
# Step 4 — Inspect request submission
# =============================================================================

print_section("Step 4 — Inspect _add_completion_requests()")

# TODO:
# Locate the internal method responsible for converting prompts
# into requests and submitting them into the engine.
#
# Hint:
# The method name was discovered in Step 3.
#
request_submission_method = LLM._add_completion_requests

# TODO:
# Find the source file.
request_submission_file = inspect.getfile(request_submission_method)

print("Request submission method source file:")
print(request_submission_file)


# TODO:
# Print the source code.
request_submission_source = inspect.getsource(request_submission_method)

print(request_submission_source)


Step 4 — Inspect _add_completion_requests()
Request submission method source file:
/usr/local/lib/python3.13/dist-packages/vllm/entrypoints/offline_utils.py
    def _add_completion_requests(
        self,
        prompts: PromptType | Sequence[PromptType],
        params: SamplingParams
        | PoolingParams
        | Sequence[SamplingParams | PoolingParams],
        *,
        use_tqdm: bool | Callable[..., tqdm] = True,
        lora_request: Sequence[LoRARequest] | LoRARequest | None = None,
        priority: list[int] | None = None,
        tokenization_kwargs: dict[str, Any] | None = None,
        mm_processor_kwargs: dict[str, Any] | None = None,
    ) -> list[str]:
        seq_prompts = prompt_to_seq(prompts)
        seq_params = self._params_to_seq(params, len(seq_prompts))
        seq_lora_requests = self._lora_request_to_seq(lora_request, len(seq_prompts))
        seq_priority = self._priority_to_seq(priority, len(seq_prompts))

        return self._render_and_add_requests(

In [19]:
# =============================================================================
# Step 5 — Inspect _render_and_add_requests()
# =============================================================================

print_section("Step 5 — Inspect _render_and_add_requests()")

# TODO:
# Locate the method that receives preprocessed prompts and adds requests
# into the engine.
#
# Hint:
# It was called at the end of _add_completion_requests().
#
render_add_method = LLM._render_and_add_requests

# TODO:
# Find its source file.
render_add_file = inspect.getfile(render_add_method)

print("Source file:")
print(render_add_file)

# TODO:
# Print the source code.
render_add_source = inspect.getsource(render_add_method)

print(render_add_source)


Step 5 — Inspect _render_and_add_requests()
Source file:
/usr/local/lib/python3.13/dist-packages/vllm/entrypoints/offline_utils.py
    def _render_and_add_requests(
        self,
        prompts: Iterable[EngineInput],
        params: Sequence[SamplingParams | PoolingParams],
        *,
        lora_requests: Sequence[LoRARequest | None] | None = None,
        priorities: Sequence[int] | None = None,
    ) -> list[str]:
        added_request_ids: list[str] = []

        try:
            for i, prompt in enumerate(prompts):
                request_id = self._add_request(
                    prompt,
                    params[i],
                    lora_request=self._resolve_mm_lora(
                        prompt,
                        None if lora_requests is None else lora_requests[i],
                    ),
                    priority=0 if priorities is None else priorities[i],
                )
                added_request_ids.append(request_id)
        except Exception as e:


In [20]:
# =============================================================================
# Step 6 — Inspect _add_request()
# =============================================================================

print_section("Step 6 — Inspect _add_request()")

# TODO:
# Locate the method that submits one request deeper into the engine.
#
# Questions to answer:
#   1. Where is request_id created?
#   2. Which engine method receives the request?
#   3. What arguments are passed into the engine?
#
add_request_method = LLM._add_request

# TODO:
# Find the source file.
add_request_file = inspect.getfile(add_request_method)

print("Source file:")
print(add_request_file)

# TODO:
# Print the source code.
add_request_source = inspect.getsource(add_request_method)

print(add_request_source)


Step 6 — Inspect _add_request()
Source file:
/usr/local/lib/python3.13/dist-packages/vllm/entrypoints/offline_utils.py
    def _add_request(
        self,
        prompt: EngineInput,
        params: SamplingParams | PoolingParams,
        lora_request: LoRARequest | None = None,
        priority: int = 0,
    ) -> str:
        if isinstance(params, SamplingParams):
            # We only care about the final output
            params.output_kind = RequestOutputKind.FINAL_ONLY

        request_id = str(next(self.request_counter))

        return self.llm_engine.add_request(
            request_id,
            prompt,
            params,
            lora_request=lora_request,
            priority=priority,
        )



In [21]:
# =============================================================================
# Step 7 — Inspect LLMEngine.add_request()
# =============================================================================

print_section("Step 7 — Inspect LLMEngine.add_request()")

# TODO:
# Identify the class of self.llm_engine.
#
# Questions:
#   1. What class owns add_request()?
#   2. Where is add_request() implemented?
#   3. What transformation happens before the request reaches EngineCore?
#

from vllm.engine.llm_engine import LLMEngine

engine_add_method = LLMEngine.add_request

# TODO:
# Locate the source file.
engine_add_file = inspect.getfile(engine_add_method)

print("Source file:")
print(engine_add_file)

# TODO:
# Print the source code.
engine_add_source = inspect.getsource(engine_add_method)

print(engine_add_source)


Step 7 — Inspect LLMEngine.add_request()
Source file:
/usr/local/lib/python3.13/dist-packages/vllm/v1/engine/llm_engine.py
    def add_request(
        self,
        request_id: str,
        prompt: EngineCoreRequest | PromptType | EngineInput,
        params: SamplingParams | PoolingParams,
        arrival_time: float | None = None,
        lora_request: LoRARequest | None = None,
        tokenization_kwargs: dict[str, Any] | None = None,
        trace_headers: Mapping[str, str] | None = None,
        priority: int = 0,
        session_id: str | None = None,
        prompt_text: str | None = None,
    ) -> str:
        # Validate the request_id type.
        if not isinstance(request_id, str):
            raise TypeError(f"request_id must be a string, got {type(request_id)}")

        # Process raw inputs into the request.
        if isinstance(prompt, EngineCoreRequest):
            logger.warning_once(
                "Passing EngineCoreRequest to LLMEngine.generate() and .add_requ

In [22]:
# =============================================================================
# Step 8 — Inspect InputProcessor.process_inputs()
# =============================================================================

print_section("Step 8 — Inspect InputProcessor.process_inputs()")

# TODO:
# Identify the class that owns self.input_processor.
#
# Questions:
#   1. What does process_inputs() return?
#   2. Where is EngineCoreRequest created?
#   3. Which fields are copied into that request?
#
from vllm.v1.engine.input_processor import InputProcessor

input_processor = InputProcessor

# TODO:
# Find the class / method source file.
input_processor_method = InputProcessor.process_inputs
input_processor_file = inspect.getfile(input_processor_method)

print("InputProcessor.process_inputs() source file:")
print(input_processor_file)

# TODO:
# Print the source code.
input_processor_source = inspect.getsource(input_processor_method)


print(input_processor_source)


Step 8 — Inspect InputProcessor.process_inputs()
InputProcessor.process_inputs() source file:
/usr/local/lib/python3.13/dist-packages/vllm/v1/engine/input_processor.py
    def process_inputs(
        self,
        request_id: str,
        prompt: PromptType | EngineInput,
        params: SamplingParams | PoolingParams,
        supported_tasks: tuple[SupportedTask, ...],
        arrival_time: float | None = None,
        lora_request: LoRARequest | None = None,
        tokenization_kwargs: dict[str, Any] | None = None,
        trace_headers: Mapping[str, str] | None = None,
        priority: int = 0,
        data_parallel_rank: int | None = None,
        resumable: bool = False,
        session_id: str | None = None,
    ) -> EngineCoreRequest:
        self._validate_params(params, supported_tasks)
        self._validate_lora(lora_request)

        parallel_config = self.vllm_config.parallel_config
        dp_size = parallel_config.data_parallel_size
        dp_local_size = parallel_co

In [23]:
# =============================================================================
# Step 9 — Trace EngineCoreRequest into the core client
# =============================================================================

print_section("Step 9 — Trace EngineCoreRequest into the core client")

from vllm.v1.engine.llm_engine import LLMEngine

# TODO:
# Inspect the complete implementation of LLMEngine.add_request().
#
# Questions:
#   1. After InputProcessor.process_inputs() returns EngineCoreRequest,
#      which object receives the request next?
#
#   2. Is the request also registered with an output processor?
#
#   3. Which method actually forwards the EngineCoreRequest toward EngineCore?
#
engine_add_method = LLMEngine.add_request

# TODO:
# Find the source file.
engine_add_file = inspect.getfile(engine_add_method)

print("LLMEngine.add_request() source file:")
print(engine_add_file)

# TODO:
# Retrieve and print the complete source code.
engine_add_source = inspect.getsource(engine_add_method)

print(engine_add_source)


Step 9 — Trace EngineCoreRequest into the core client
LLMEngine.add_request() source file:
/usr/local/lib/python3.13/dist-packages/vllm/v1/engine/llm_engine.py
    def add_request(
        self,
        request_id: str,
        prompt: EngineCoreRequest | PromptType | EngineInput,
        params: SamplingParams | PoolingParams,
        arrival_time: float | None = None,
        lora_request: LoRARequest | None = None,
        tokenization_kwargs: dict[str, Any] | None = None,
        trace_headers: Mapping[str, str] | None = None,
        priority: int = 0,
        session_id: str | None = None,
        prompt_text: str | None = None,
    ) -> str:
        # Validate the request_id type.
        if not isinstance(request_id, str):
            raise TypeError(f"request_id must be a string, got {type(request_id)}")

        # Process raw inputs into the request.
        if isinstance(prompt, EngineCoreRequest):
            logger.warning_once(
                "Passing EngineCoreRequest 

In [24]:
# =============================================================================
# Step 10 — Inspect EngineCore.add_request()
# =============================================================================

print_section("Step 10 — Inspect EngineCore.add_request()")

# TODO:
# Identify the class behind self.engine_core and inspect add_request().
#
# Questions:
#   1. Does EngineCore directly own the Scheduler?
#   2. Is EngineCore.add_request() synchronous or forwarded through a client?
#   3. Where does Scheduler.add_request() get called?
#
from vllm.v1.engine.core import EngineCore
# engine_core_object = EngineCore

# TODO:
# Identify the class of the engine core object.
engine_core_class = EngineCore

print("EngineCore class:")
print(engine_core_class)

# TODO:
# Get the add_request() method from the class.
engine_core_add_method = EngineCore.add_request

# TODO:
# Find its source file.
engine_core_add_file = inspect.getfile(engine_core_add_method)

print("EngineCore.add_request() source file:")
print(engine_core_add_file)

# TODO:
# Print the source code.
engine_core_add_source = inspect.getsource(engine_core_add_method)

print(engine_core_add_source)


Step 10 — Inspect EngineCore.add_request()
EngineCore class:
<class 'vllm.v1.engine.core.EngineCore'>
EngineCore.add_request() source file:
/usr/local/lib/python3.13/dist-packages/vllm/v1/engine/core.py
    def add_request(self, request: Request, request_wave: int = 0):
        """Add request to the scheduler.

        `request_wave`: indicate which wave of requests this is expected to
        belong to in DP case
        """
        # Validate the request_id type.
        if not isinstance(request.request_id, str):
            raise TypeError(
                f"request_id must be a string, got {type(request.request_id)}"
            )

        if pooling_params := request.pooling_params:
            supported_pooling_tasks = [
                task for task in self.get_supported_tasks() if task in POOLING_TASKS
            ]

            if pooling_params.task not in supported_pooling_tasks:
                raise ValueError(
                    f"Unsupported task: {pooling_params.task

In [25]:
# =============================================================================
# Step 11 — Inspect Scheduler.add_request()
# =============================================================================

print_section("Step 11 — Inspect Scheduler.add_request()")

from vllm.v1.core.sched.scheduler import Scheduler

# TODO:
# Inspect Scheduler.add_request().
#
# Questions:
#   1. Which queue does a new request enter?
#   2. Is request priority considered here?
#   3. Does add_request() perform execution, or only admission?
#

scheduler_add_method = Scheduler.add_request

# TODO:
# Locate its source file.
scheduler_add_file = inspect.getfile(scheduler_add_method)

print("Scheduler.add_request() source file:")
print(scheduler_add_file)

# TODO:
# Print the source code.
scheduler_add_source = inspect.getsource(scheduler_add_method)

print(scheduler_add_source)


Step 11 — Inspect Scheduler.add_request()
Scheduler.add_request() source file:
/usr/local/lib/python3.13/dist-packages/vllm/v1/core/sched/scheduler.py
    def add_request(self, request: Request) -> None:
        existing = self.requests.get(request.request_id)
        if existing is not None:
            update = StreamingUpdate.from_request(request)
            if existing.status != RequestStatus.WAITING_FOR_STREAMING_REQ:
                assert existing.streaming_queue is not None, "duplicate request id"
                # Queue next input chunk (or finished sentinel).
                existing.streaming_queue.append(update)
            elif update is not None:
                # Commence next input chunk.
                self._update_request_as_session(existing, update)
            else:
                # Streaming-input session finished.
                self.finish_requests(request.request_id, RequestStatus.FINISHED_ABORTED)
        else:
            if request.resumable:
           

In [26]:
# =============================================================================
# Step 12 — Inspect Scheduler.schedule()
# =============================================================================

print_section("Step 12 — Inspect Scheduler.schedule()")

# TODO:
# Inspect Scheduler.schedule().
#
# Questions:
#   1. Does it examine waiting requests, running requests, or both?
#   2. How does it decide how many tokens to schedule?
#   3. What object does it return for the next engine step?
#
scheduler_schedule_method = Scheduler.schedule

# TODO:
# Locate the source file.
scheduler_schedule_file = inspect.getfile(scheduler_schedule_method)

print("Scheduler.schedule() source file:")
print(scheduler_schedule_file)

# TODO:
# Print the source code.
scheduler_schedule_source = inspect.getsource(scheduler_schedule_method)

print(scheduler_schedule_source)


Step 12 — Inspect Scheduler.schedule()
Scheduler.schedule() source file:
/usr/local/lib/python3.13/dist-packages/vllm/v1/core/sched/scheduler.py
    def schedule(self, throttle_prefills: bool = False) -> SchedulerOutput:
        self.current_step += 1
        # NOTE(woosuk) on the scheduling algorithm:
        # There's no "decoding phase" nor "prefill phase" in the scheduler.
        # Each request just has the num_computed_tokens and
        # num_tokens_with_spec. num_tokens_with_spec =
        # len(prompt_token_ids) + len(output_token_ids) + len(spec_token_ids).
        # At each step, the scheduler tries to assign tokens to the requests
        # so that each request's num_computed_tokens can catch up its
        # num_tokens_with_spec. This is general enough to cover
        # chunked prefills, prefix caching, speculative decoding,
        # and the "jump decoding" optimization in the future.

        scheduled_new_reqs: list[Request] = []
        scheduled_resumed_reqs: list[R

## Scheduler Flow

```text
Scheduler.schedule()
        ↓
Create token budget
        ↓
Schedule RUNNING requests
        ↓
Calculate num_new_tokens
        ↓
Allocate KV cache slots
        ↓
Schedule WAITING requests
        ↓
WAITING → RUNNING
        ↓
Record scheduled token counts
        ↓
Build SchedulerOutput
        ↓
ModelRunner / Execution

## Scheduler Execution Flow

```text
User Prompt
    ↓
LLM.generate()
    ↓
_run_completion()
    ↓
_add_completion_requests()
    ↓
_render_and_add_requests()
    ↓
_add_request()
    ↓
LLMEngine.add_request()
    ↓
InputProcessor.process_inputs()
    ↓
EngineCoreRequest
    ↓
EngineCore.add_request()
    ↓
Scheduler.add_request()
    ↓
WAITING
    ↓
Scheduler.schedule()
    ↓
Token-budget decision
    ↓
KV-cache allocation
    ↓
RUNNING
    ↓
SchedulerOutput
    ↓
Model Execution